# Session 2: the mathematics of levelling and of the total station
**A. Real-world problem.** A real levelling run is written in a *levelling book*, a table of
readings. A mistake in the table ruins every level after it, so surveyors use two methods and
an arithmetic check. A *total station* adds the second half of surveying: it measures a slope
distance and angles, and the computer turns them into easting and northing. Finally, how far
apart should the staff positions be, so that small reading errors do not pile up over 1 km?

**Plan.** (1) reduce a levelling book two ways, (2) the arithmetic check, (3) horizontal
distance from a slope distance, (4) easting and northing from bearing and distance,
(5) the square-root law for accumulated error.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# A levelling book: each row is (kind, reading in metres). BS = backsight, IS = intermediate
# sight, FS = foresight. A foresight followed by a backsight is the same point (a change point).
BOOK = [("BS", 1.425), ("IS", 1.870), ("IS", 2.315), ("FS", 2.905),
        ("BS", 1.150), ("IS", 0.980), ("FS", 0.640)]
RL_BM = 100.000   # reduced level of the first point (a benchmark)

## Step 1: height of instrument (collimation) method
At each set-up, the height of instrument is HI = RL of the backsight point + backsight.
Every other point seen from that set-up has RL = HI - reading.

In [ ]:
def hi_method(rows, rl0=RL_BM):
    """Reduced level for every row. Returns (rls, hi_values)."""
    rls, his, hi = [], [], None
    for kind, r in rows:
        if kind == "BS":
            rl = rl0 if not rls else rls[-1]     # first point, or the change point just left
            hi = rl + r
            his.append(hi)
        else:
            rl = hi - r
            his.append(None)
        rls.append(rl)
    return rls, his

## Step 2: rise-and-fall method, and the arithmetic check
Compare each reading with the one before it in the same set-up: rise = previous - current
(positive is a rise, negative a fall), RL = previous RL + rise.

**Arithmetic check:** sum of backsights - sum of foresights = last RL - first RL.
Warning: the check only proves the arithmetic along the chain of backsights and foresights.
It cannot find a wrong intermediate sight, because no later level uses it.

In [ ]:
def rise_fall(rows, rl0=RL_BM):
    """Reduced level for every row by rise and fall. Returns (rls, rises); rises[0 or CP] is None."""
    rls, rises, prev = [], [], None
    for kind, r in rows:
        if kind == "BS":
            rl = rl0 if not rls else rls[-1]
            rises.append(None)
        else:
            rise = prev - r
            rl = rls[-1] + rise
            rises.append(rise)
        rls.append(rl)
        prev = r
    return rls, rises


def arithmetic_check(rows, rls):
    """Returns (sum_bs, sum_fs, last_rl - first_rl). The first two differ by the third if all is well."""
    sb = sum(r for k, r in rows if k == "BS")
    sf = sum(r for k, r in rows if k == "FS")
    return sb, sf, rls[-1] - rls[0]

## Step 3: slope distance to horizontal distance
A tape or total station measures along the slope: S. A map needs the horizontal distance.
With elevation angle a (above horizontal): D = S cos(a) and height change V = S sin(a).

In [ ]:
def horizontal_distance(S, a_deg):
    return S * np.cos(np.radians(a_deg))


def vertical_component(S, a_deg):
    return S * np.sin(np.radians(a_deg))

## Step 4: easting and northing from bearing and distance
A bearing is an angle measured clockwise from grid north. East is E, north is N:

    E = E0 + D sin(bearing)        N = N0 + D cos(bearing)

(sin goes with east because bearing 90 degrees points east.) The reverse problem, the
*inverse*, gives bearing = atan2(dE, dN) and distance = sqrt(dE^2 + dN^2).

In [ ]:
def total_station_point(E0, N0, rl0, ih, th, S, a_deg, bearing_deg):
    """One shot: station at (E0, N0, rl0), instrument height ih, prism height th, slope distance S,
    elevation angle a_deg, bearing bearing_deg. Returns dict with D, E, N and RL of the ground point."""
    D = float(horizontal_distance(S, a_deg))
    b = np.radians(bearing_deg)
    return {"D": D, "E": E0 + D * np.sin(b), "N": N0 + D * np.cos(b),
            "RL": rl0 + ih + float(vertical_component(S, a_deg)) - th}


def inverse(E1, N1, E2, N2):
    """Bearing (degrees, 0 to 360) and distance from point 1 to point 2."""
    dE, dN = E2 - E1, N2 - N1
    return float(np.degrees(np.arctan2(dE, dN)) % 360.0), float(np.hypot(dE, dN))

## Step 5: the square-root law for levelling error
Suppose each set-up adds an independent random error with standard deviation s (here 2 mm).
After n set-ups the errors add like the sides of a random walk, not like a plain sum:

    error(n) = s sqrt(n)

For a line of length L with the staffs spaced d apart (one set-up per d), n = L / d.

In [ ]:
def levelling_error(n, s):
    return s * np.sqrt(n)


def max_setups(s, limit):
    """Largest whole number of set-ups with s sqrt(n) strictly below limit."""
    n = int(np.floor((limit / s) ** 2))
    return n - 1 if abs(levelling_error(n, s) - limit) < 1e-12 else n


def min_spacing(total_length, s, limit):
    """Smallest staff spacing (same unit as total_length) that keeps the error below the limit."""
    return total_length / max_setups(s, limit)


def simulate_levelling(n, s, trials=20000, seed=1):
    """Monte Carlo: standard deviation of the accumulated error of n set-ups."""
    rng = np.random.default_rng(seed)
    return float(rng.normal(0.0, s, size=(trials, n)).sum(axis=1).std())

## Step 6: see it
Left: the error grows slowly (square root) while a one-sided systematic error would grow
straight. Right: the slope distance versus the horizontal distance.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    rls_h, his = hi_method(BOOK)
    rls_r, rises = rise_fall(BOOK)
    for (k, r), a, b in zip(BOOK, rls_h, rls_r):
        print(f"{k}  {r:6.3f}   RL (HI) {a:8.3f}   RL (rise/fall) {b:8.3f}")
    sb, sf, dr = arithmetic_check(BOOK, rls_h)
    print(f"sum BS {sb:.3f} - sum FS {sf:.3f} = {sb - sf:.3f}; last RL - first RL = {dr:.3f}")

    p = total_station_point(1000.0, 5000.0, 50.0, 1.55, 1.80, 85.340, 7.5, 63.5)
    print("total station point:", {k: round(v, 3) for k, v in p.items()})

    n = np.arange(1, 61)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(n, levelling_error(n, 2.0), label="random errors: 2 mm x sqrt(n)")
    ax.plot(n, 1.0 * n, label="systematic 1 mm per set-up: 1 mm x n")
    ax.axhline(10, color="red", ls="--", label="10 mm limit")
    ax.axvline(24, color="gray", ls=":")
    ax.set_xlabel("number of set-ups n")
    ax.set_ylabel("accumulated error (mm)")
    ax.set_title("Random error grows like sqrt(n): 24 set-ups give 9.8 mm")
    ax.set_ylim(0, 40)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "error_growth.png", dpi=120)

    a = np.linspace(-30, 30, 121)
    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    ax2.plot(a, horizontal_distance(100.0, a), label="horizontal D = S cos(a)")
    ax2.plot(a, vertical_component(100.0, a), label="vertical V = S sin(a)")
    ax2.axhline(100.0, color="gray", ls=":")
    ax2.set_xlabel("elevation angle a (degrees)")
    ax2.set_ylabel("distance for S = 100 m (m)")
    ax2.set_title("Slope distance 100 m split into D and V")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "slope_split.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the second backsight in `BOOK` to 1.250 and re-run both methods. Predict first which levels change and what the arithmetic check shows.
2. **Stretch.** Write `closing_error(rows, rl_known_end)` that returns the difference between the computed last level and a known level, and use it on `BOOK` with a known end level of 99.030.
3. **Explain.** Explain why 24 set-ups is allowed but 25 set-ups is not when s = 2 mm and the limit is 10 mm.

## Assessment
- **A. Conceptual.** A student adds a misread intermediate sight (0.100 m too big) to the levelling book and finds the arithmetic check still passes. Is the check useless? What does it prove and what does it not?
- **B. Numerical.** A tape along a slope reads S = 120.50 m at an elevation angle of 12 degrees. Find D and V. Then, from E 1000, N 5000, find the point 50 m away at bearing 245 degrees.
- **C. Graph and simulation interpretation.** In the graph of D and V against the angle for S = 100 m, how much shorter than S is D at 10 degrees and at 25 degrees? What does this say about when you can ignore the slope?
- **D. Design and optimisation challenge.** Design: a 4 km levelling line must have a random error below 10 mm, and each set-up has s = 1.5 mm. How many set-ups at most, and what spacing is that? Choose a practical spacing of whole tens of metres and give the resulting error. Mention one other limit on the spacing.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    rh, his = L.hi_method(L.BOOK)
    rr, rises = L.rise_fall(L.BOOK)
    want = [100.000, 99.555, 99.110, 98.520, 98.520, 98.690, 99.030]
    checks.append(("worked example: height of instrument method gives 99.555, 99.110, 98.520, 98.690, 99.030",
                   np.allclose(rh, want, atol=1e-9)))
    checks.append(("worked example: first height of instrument is 101.425 m and the second 99.670 m",
                   abs(his[0] - 101.425) < 1e-9 and abs(his[4] - 99.670) < 1e-9))
    checks.append(("rise and fall gives the same levels as height of instrument", np.allclose(rh, rr, atol=1e-9)))
    checks.append(("rise and fall: first intermediate point is a fall of 0.445 m", abs(rises[1] + 0.445) < 1e-9))
    sb, sf, dr = L.arithmetic_check(L.BOOK, rh)
    checks.append(("worked example: sum BS 2.575 - sum FS 3.545 = -0.970 = last RL - first RL",
                   abs(sb - 2.575) < 1e-9 and abs(sf - 3.545) < 1e-9 and abs((sb - sf) - dr) < 1e-9 and abs(dr + 0.970) < 1e-9))
    bad = list(rh)
    bad[-1] += 0.010
    checks.append(("an arithmetic slip of 0.010 m in the last level breaks the check",
                   abs((sb - sf) - (bad[-1] - bad[0])) > 0.009))
    book2 = list(L.BOOK)
    book2[1] = ("IS", 1.970)           # misread intermediate sight (0.100 m too big)
    r2, _ = L.hi_method(book2)
    s1, s2, d2 = L.arithmetic_check(book2, r2)
    checks.append(("a wrong intermediate sight still passes the check but its own level is wrong by 0.100 m",
                   abs((s1 - s2) - d2) < 1e-9 and abs(r2[1] - rh[1] + 0.100) < 1e-9))
    checks.append(("worked example: S = 85.340 m at 7.5 degrees gives D = 84.610 m",
                   abs(L.horizontal_distance(85.340, 7.5) - 84.610) < 5e-4))
    p = L.total_station_point(1000.0, 5000.0, 50.0, 1.55, 1.80, 85.340, 7.5, 63.5)
    checks.append(("worked example: bearing 63.5 gives E = 1075.720 and N = 5037.753",
                   abs(p["E"] - 1075.720) < 5e-4 and abs(p["N"] - 5037.753) < 5e-4))
    checks.append(("worked example: the prism point is at RL 60.889 m", abs(p["RL"] - 60.889) < 5e-4))
    b, d = L.inverse(1000.0, 5000.0, p["E"], p["N"])
    checks.append(("the inverse returns the bearing 63.5 degrees and distance 84.610 m",
                   abs(b - 63.5) < 1e-9 and abs(d - 84.610) < 5e-4))
    checks.append(("hand calculation: S = 120.50 m at 12 degrees gives D = 117.87 m and V = 25.05 m",
                   abs(L.horizontal_distance(120.5, 12) - 117.87) < 5e-3 and abs(L.vertical_component(120.5, 12) - 25.05) < 5e-3))
    checks.append(("hand calculation: 50 m at bearing 245 degrees lands at E 954.68, N 4978.87",
                   abs(1000 + 50 * np.sin(np.radians(245)) - 954.68) < 5e-3 and abs(5000 + 50 * np.cos(np.radians(245)) - 4978.87) < 5e-3))
    b0, d0 = L.inverse(0.0, 0.0, 10.0, 0.0)
    checks.append(("edge: due east is bearing 90 degrees and due south is 180", abs(b0 - 90) < 1e-9 and abs(L.inverse(0, 0, 0, -5)[0] - 180) < 1e-9))
    checks.append(("edge: due west gives bearing 270 degrees (not -90)", abs(L.inverse(0, 0, -3, 0)[0] - 270) < 1e-9))
    checks.append(("edge: a level slope (0 degrees) gives D = S", abs(L.horizontal_distance(77.7, 0.0) - 77.7) < 1e-12))
    checks.append(("a 10 degree slope shortens the distance by 1.5 percent, a 25 degree slope by 9.4 percent",
                   abs((1 - np.cos(np.radians(10))) * 100 - 1.5) < 0.05 and abs((1 - np.cos(np.radians(25))) * 100 - 9.4) < 0.05))
    checks.append(("worked example: 2 mm x sqrt(24) = 9.80 mm, 2 mm x sqrt(25) = 10.00 mm (not below 10)",
                   abs(L.levelling_error(24, 2.0) - 9.798) < 5e-4 and abs(L.levelling_error(25, 2.0) - 10.0) < 1e-12))
    checks.append(("worked example: at most 24 set-ups, so spacing at least 41.7 m over 1 km",
                   L.max_setups(2.0, 10.0) == 24 and abs(L.min_spacing(1000.0, 2.0, 10.0) - 41.667) < 5e-3))
    checks.append(("a systematic 1 mm per set-up over 24 set-ups reaches 24 mm, far above 9.8 mm", 1.0 * 24 > 2 * L.levelling_error(24, 2.0)))
    checks.append(("Monte Carlo (seeded) agrees with 2 sqrt(24) within 2 percent",
                   abs(L.simulate_levelling(24, 2.0) / L.levelling_error(24, 2.0) - 1.0) < 0.02))
    checks.append(("design: 4 km with 1.5 mm per set-up allows 44 set-ups; 40 set-ups give 9.49 mm",
                   L.max_setups(1.5, 10.0) == 44 and abs(L.levelling_error(40, 1.5) - 9.487) < 5e-4))
    checks.append(("design: 4 km with 40 set-ups means staffs 100 m apart; the minimum spacing is 90.9 m",
                   abs(4000.0 / 40 - 100.0) < 1e-9 and abs(L.min_spacing(4000.0, 1.5, 10.0) - 90.909) < 5e-3))
    checks.append(("worked example: V = 85.340 sin 7.5 degrees = 11.139 m; (10 / 1.5)^2 = 44.4",
                   abs(L.vertical_component(85.340, 7.5) - 11.139) < 5e-4 and abs((10 / 1.5) ** 2 - 44.4) < 0.05))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
